In [3]:
import pandas as pd

#Импортируем необходимые файлы
cases = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\cases.csv')
contacts = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\contacts.csv')
payments = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\payments.csv')

In [2]:
cases.head()

,case_id,account_id,customer_id,start_date,dpd_bucket,strategy_name,principal_amount
0,1,50000,80000,2026-02-27,31-60,late_stage,68845.38
1,2,50001,80001,2026-02-18,1-30,late_stage,55274.70
2,3,50002,80002,2026-02-02,31-60,mixed,52909.61
3,4,50003,80003,2026-03-25,61-90,sms_first,64609.93
4,5,50004,80004,2026-01-24,1-30,sms_first,53838.74


In [3]:
contacts.head()

,event_id,case_id,event_dttm,channel,contact_result
0,4505,1703,2026-03-15 17:11:00,sms,RPC
1,2588,983,2026-02-24 22:54:00,sms,Promise
2,6466,2419,2026-04-01 13:54:00,call,Promise
3,5671,2118,2026-02-08 18:29:00,call,No answer
4,4414,1669,2026-01-11 15:57:00,call,No answer


In [4]:
payments.head()

,payment_id,account_id,payment_dttm,payment_amount
0,1,50002,2026-02-10 14:54:00,6397.25
1,2,50005,2026-02-03 16:03:00,25241.16
2,3,50006,2026-01-31 15:09:00,17364.78
3,4,50007,2026-02-04 08:36:00,13084.38
4,5,50008,2026-03-03 12:33:00,0.00


In [5]:
def build_collection_summary(cases, contacts, payments):
    # Создадим копии входных таблиц
    cases = cases.copy()
    contacts = contacts.copy()
    payments = payments.copy()

    # Приведем даты к datetime
    cases['start_date'] = pd.to_datetime(cases['start_date'])
    contacts['event_dttm'] = pd.to_datetime(contacts['event_dttm'])
    payments['payment_dttm'] = pd.to_datetime(payments['payment_dttm'])

    # Удалим дубли в contacts по event_id - оставляем запись с максимальным event_dttm
    contacts = (contacts.sort_values(['event_id', 'event_dttm']).drop_duplicates(subset=['event_id'], keep='last'))

    # Исключим невалидные платежей
    payments = payments[payments['payment_amount'] > 0].copy()

    # Добавим границу 7-дневного окна: start_date и start_date + 6 дней
    cases['end_date_7d'] = cases['start_date'] + pd.Timedelta(days=6)

    # Посчитаем кол-во уникалькых кейсов в каждом бакете
    total_cases = (cases.groupby('dpd_bucket', dropna=False)['case_id'].nunique().rename('total_cases'))

    # Объединим таблицы contacts и cases - чтобы для каждого контатка была информацию о кейсе
    contacts_full = contacts.merge(cases[['case_id', 'dpd_bucket', 'start_date', 'end_date_7d']],on='case_id',how='inner')

    # Оставим только те контакты, которые произошли в течение 7 дней после старта кейса
    actual_contacts = contacts_full[(contacts_full['event_dttm'] >= contacts_full['start_date']) & 
    (contacts_full['event_dttm'] <= contacts_full['end_date_7d'])]

    # Выясним, сколько кейсов в каждом бакете имели хотя бы один контакт в первые 7 дней
    contacted_cases_7d = (actual_contacts.groupby('dpd_bucket', dropna=False)['case_id'].nunique().rename('contacted_cases_7d'))

    # Объединим таблицы payments и cases - чтобы для каждого платежа была информацию о кейсе
    payments_full = payments.merge(cases[['case_id', 'account_id', 'dpd_bucket', 'start_date', 'end_date_7d']],on='account_id',how='inner')

    # Оставим только те платежи, которые произошли в течение 7 дней после старта кейса
    actual_payments = payments_full[(payments_full['payment_dttm'] >= payments_full['start_date']) &
        (payments_full['payment_dttm'] <= payments_full['end_date_7d'])]

    # Выясним, сколько кейсов в каждом бакете заплатили в первые 7 дней
    paid_cases_7d = (actual_payments.groupby('dpd_bucket', dropna=False)['case_id'].nunique().rename('paid_cases_7d'))

    # Вычислим, сколько денег было собрано в каждом бакете в первые 7 дней
    collected_amount_7d = (actual_payments.groupby('dpd_bucket', dropna=False)['payment_amount'].sum().rename('collected_amount_7d'))

    # Создадим витрину
    result = pd.concat([total_cases, contacted_cases_7d, paid_cases_7d, collected_amount_7d],axis=1).fillna({'contacted_cases_7d': 0,'paid_cases_7d': 0,'collected_amount_7d': 0,})

    # Приведем значения к типу int
    result['contacted_cases_7d'] = result['contacted_cases_7d'].astype(int)
    result["paid_cases_7d"] = result["paid_cases_7d"].astype(int)

    # Посчитаем средние платежи для каждого кейса в первые 7 дней
    result['avg_payment_per_paid_case_7d'] = (result['collected_amount_7d'] / result['paid_cases_7d'].replace(0, pd.NA)).fillna(0)

    # Сортировка бакетов в нужном порядке
    bucket_order = ['1-30', '31-60', '61-90', '90+']
    result = result.reset_index()
    result['dpd_bucket'] = pd.Categorical(result['dpd_bucket'], categories=bucket_order, ordered=True)
    result = result.sort_values('dpd_bucket').reset_index(drop=True)

    return result

In [6]:
# Выведем витрину на экран
print(build_collection_summary(cases, contacts, payments))

  dpd_bucket  total_cases  contacted_cases_7d  paid_cases_7d  \
0       1-30         1086                 492            323   
1      31-60          755                 419            170   
2      61-90          489                 340             90   
3        90+          270                 196             20   

   collected_amount_7d  avg_payment_per_paid_case_7d  
0           3367920.36                  10426.998019  
1           2349000.61                  13817.650647  
2           1317981.54                  14644.239333  
3            345482.94                  17274.147000  
